# Imports

In [1]:
from WS_Mdl.core import *

In [2]:
import datetime as DT
import pandas as pd

# Prep

In [8]:
MdlN = 'NBr70'
MdlN_B = 'NBr68'

In [9]:
M = Mdl_N(MdlN)
MB = Mdl_N(MdlN_B)

Could not determine imod version from Sim/NBr70 folder.
Proceeding with the assumption that it's imod_python.


In [59]:
Pa_SFR_Out = MB.Pa.Sim_In / f'{MB.MdlN}.SFR6.obs.output.csv'
DF = pd.read_csv(Pa_SFR_Out)

In [60]:
DF['date'] = pd.to_datetime(DF['time'], unit='d', origin=pd.Timestamp(MB.INI.SDATE)) - DT.timedelta(days=1) # Convert time to date

In [61]:
DF['month'] = DF['date'].dt.month

In [62]:
DF = DF[['date', 'month'] + [i for i in DF.columns if 'STG' in i]] # Only keep stage columns

In [63]:
DF.loc['AVG_Stg_summer'] = DF.loc[(DF['month']>3) & (DF['month']<10)].copy().mean(axis='index')
DF.loc['AVG_Stg_winter'] = DF.loc[(DF['month']>=10) | (DF['month']<=3)].copy().mean(axis='index')
DF.loc['AVG_Stg_winter_m_summer'] = DF.loc['AVG_Stg_winter'] - DF.loc['AVG_Stg_summer']

In [64]:
DF.drop([i for i in DF.index if i not in ['AVG_Stg_summer', 'AVG_Stg_winter', 'AVG_Stg_winter_m_summer']], inplace=True)

In [65]:
DF = DF.drop(['date', 'month'], axis='columns').T

In [66]:
DF

,AVG_Stg_summer,AVG_Stg_winter,AVG_Stg_winter_m_summer
STG_L7_R282_C393,18.800000,18.800000,0.000000
STG_L7_R281_C393,18.800000,18.800000,0.000000
STG_L7_R281_C392,18.800000,18.800000,0.000000
STG_L7_R281_C391,18.800000,18.800000,0.000000
STG_L7_R281_C390,18.800000,18.800000,0.000000
...,...,...,...
STG_L5_R24_C17,0.079921,0.127964,0.048043
STG_L5_R24_C16,0.080198,0.128236,0.048038
STG_L5_R23_C16,-0.078371,-0.072611,0.005760
STG_L5_R23_C15,-0.621920,-0.574591,0.047329


In [67]:
DF[['L', 'R', 'C']] = DF.index.to_series().str.extract(r'L(\d+)_R(\d+)_C(\d+)').astype(int)

In [68]:
DF = DF.ws.Calc_XY(MB.Xmin, MB.Ymax, MB.cellsize)

In [74]:
DA = DF.set_index(['y', 'x'])[['AVG_Stg_summer', 'AVG_Stg_winter', 'AVG_Stg_winter_m_summer']].to_xarray()

In [75]:
import imod

In [76]:
Pa_Out = MB.Pa.PoP_Out_MdlN / f'SFR/Stg/SFR_AVG_Stg_summer_{MB.MdlN}.IDF'
imod.idf.save(Pa_Out, DA['AVG_Stg_summer'])

In [77]:
Pa_Out = MB.Pa.PoP_Out_MdlN / f'SFR/Stg/SFR_AVG_Stg_winter_{MB.MdlN}.IDF'
imod.idf.save(Pa_Out, DA['AVG_Stg_winter'])

In [78]:
Pa_Out = MB.Pa.PoP_Out_MdlN / f'SFR/Stg/SFR_AVG_Stg_winter_m_summer_{MB.MdlN}.IDF'
imod.idf.save(Pa_Out, DA['AVG_Stg_winter_m_summer'])